# Lesson 1

## Set up
You need to be in a virtual environment that has python-dotenv and openai installed

In [4]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [5]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

Start with a direct API call to OpenAI

In [6]:
# helper function that calls the chatGPT model with a given prompt
# setting temperature to 0 to get a deterministic response (reduces the randomness of the response)
def get_completion(prompt, model=llm_model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=model,
    messages=messages,
    temperature=0)
    return response.choices[0].message.content


In [7]:
get_completion("What is 1+1?")

'1+1 equals 2.'

Querying with a more complex prompt - asking chatgpt to translate an email

In [8]:
# email to translate
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

# style to translate to 
style = """American English \
in a calm and respectful tone
"""

# prompt to use for chatgpt
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [9]:
get_completion(prompt)

"Oh man, I'm really frustrated that my blender lid flew off and splattered my kitchen walls with smoothie! And to top it off, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now, buddy."

## Using LangChain
Doing the same thing but using langchain

In [10]:
from langchain_openai import ChatOpenAI
# ChatOpenAI is langchains extraction for chatgpt endpoint

In [11]:
chat = ChatOpenAI(temperature = 0.0, model = llm_model)
chat

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x110c03250>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x110d32cf0>, root_client=<openai.OpenAI object at 0x110b5d940>, root_async_client=<openai.AsyncOpenAI object at 0x110d323c0>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))

The above line creates a ChatOpenAI instance from LangChain that acts as a wrapper around OpenAI's chat completion API. 

Parameters:
* temperature=0.0: Sets the temperature to 0, which makes the model's responses completely deterministic (no randomness). With temperature 0, the model will always give the same response for the same input prompt.
* model=llm_model: Uses the model specified by the llm_model variable, which in your notebook is set to either "gpt-3.5-turbo" or "gpt-3.5-turbo-0301" depending on the current date.

What it creates:
The chat object is a LangChain ChatOpenAI instance that provides a standardized interface to interact with OpenAI's chat models. It handles:
* API calls to OpenAI
* Message formatting
* Response processing
* Error handling

In [12]:
# creating prompt template
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""



In [13]:
# importing langchain chat prompt template

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)


In [14]:
# you can extract the original prompt from the prompt template

prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [15]:
# you can use this to see the input variables

prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [16]:
# settting the input variables

customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

# generating the prompt

customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

# you can use the code below to see the type of the object and the content of the prompt
print(type(customer_messages))
print(type(customer_messages[0]))
print(customer_messages[0])

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>
content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [17]:
# Call the LLM to translate to the style of the customer message
customer_response = chat(customer_messages)
print(customer_response.content) # this gives you the translated text

/var/folders/7j/npjz00mn76s2k495vh_q9js00000gn/T/ipykernel_9880/3972634147.py:2: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  customer_response = chat(customer_messages)


I am really frustrated that my blender lid flew off and splattered my kitchen walls with smoothie! And to make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I need your help right now, friend.


Trying another prompt

In [18]:
# message to translate
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

# setting style
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

# generating the prompt and printing the content of the prompt
service_messages = prompt_template.format_messages(
                    style=service_style_pirate,
                    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [19]:
# sending the prompt to chatGPT
service_response = chat(service_messages)
print(service_response.content)

Ahoy there, valued customer! Regrettably, the warranty be not coverin' the cost o' cleanin' yer galley due to yer own negligence. Ye see, 'twas yer own doin' that ye forgot to secure the lid afore startin' the blender. 'Tis a tough break, indeed! Fare thee well!


## Why do we use prompt templates?

Prompts can be long and detailed so prompt templates are a useful abstraction to help you reuse good prompts when you can. LangChain also provides prompts for common operations.

LangChain supports output parsing with prompt templates. You often ask LLM's to generate an output in a certain format. LangChain library functions parse the LLM's output assuming that it will use certain keywords. An example may use Thought, Action, and Observation as keywords for the output to follow a Chain-of-Thought-Reasoning (ReAct) format.

## Output parsers

We start by defining how we would like the LLM output to look like:

In [20]:
# example: extracting information from a product review

{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [21]:
# below is the customer review (input)
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

# define the template
review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [23]:
from langchain_core.prompts import ChatPromptTemplate

# create the prompt template using the template we created in the last cell
prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [24]:
# create the message to parse to the openai end point
messages = prompt_template.format_messages(text=customer_review)
# create the openai endpoint
chat = ChatOpenAI(temperature=0.0, model=llm_model)
# call the openai endpoint
response = chat(messages)
print(response.content)

{
    "gift": true,
    "delivery_days": 2,
    "price_value": "It's slightly more expensive than the other leaf blowers out there"
}


In [26]:
# note that when we print the type of the response we see that the response is a string even though it looks like a json/dictionary
type(response.content)

str

In [27]:
# Trying to get the value of the gift
# You will get an error by running this line of code 
# because'gift' is not a dictionary
# 'gift' is a string
response.content.get('gift')

AttributeError: 'str' object has no attribute 'get'

## Parse the LLM output string into a Python dictionary

In [ ]:
# old imports
# from langchain.output_parsers import ResponseSchema
# from langchain.output_parsers import StructuredOutputParser

# new versions for imports
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers.structured import ResponseSchema

In [ ]:
# old code
# we tell it what we want it to parse by specifying these output schemas
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

# creating a list of schemas
response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

By specifying the output schema you want for your response, LangChain can actually give you the prompt.

In [35]:
# old code:
# output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

StructuredOutputParser no longer exists in the new updates of LangChain. However you cannot directly translate the code and still use a response schema because JsonOutputParser works with Pydantic models, whereas StructuredOutputParser works with ResponseSchema objects.

The issue is that JsonOutputParser doesn't have a from_response_schemas() method. Looking at the code, JsonOutputParser works with Pydantic models, not ResponseSchema objects. In order to use JsonOutputParser, you have to create a Pydantic model. 

This requires the following installs:

In [36]:
from pydantic import BaseModel, Field

## Pydantic models

A Pydantic model is a Python class that defines the structure and validation rules for data using the Pydantic library. It's like a blueprint for data with built-in validation, type checking, and serialization.
What Pydantic Does
Pydantic is a data validation library that uses Python type annotations to:
Validate data - ensures data matches expected types and constraints
Parse data - converts strings/JSON into Python objects
Serialize data - converts Python objects back to JSON/dicts
Generate schemas - creates JSON schemas for API documentation

In [ ]:
# an example of a pydantic model

from pydantic import BaseModel, Field
from typing import Optional

class User(BaseModel):
    name: str
    age: int = Field(gt=0, le=120)  # age must be 1-120
    email: str
    is_active: bool = True
    bio: Optional[str] = None

# Usage
user_data = {
    "name": "Alice",
    "age": 25,
    "email": "alice@example.com"
}

user = User(**user_data)  # Validates and creates User object
print(user.name)  # "Alice"
print(user.model_dump())  # Convert back to dict

Alice
{'name': 'Alice', 'age': 25, 'email': 'alice@example.com', 'is_active': True, 'bio': None}


## An example in the context of LangChain

Pydantic models are used for:
1. Structured Output Parsing - Define what the LLM should return:

In [38]:
class LLMResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    confidence: float = Field(ge=0, le=1, description="Confidence score 0-1")
    sources: list[str] = Field(description="List of source URLs")

# Use with JsonOutputParser
parser = JsonOutputParser(pydantic_object=LLMResponse)

2. Tool Schemas - Define function parameters for LLM tools:


In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    location: str = Field(description="City name")
    unit: str = Field(default="celsius", description="Temperature unit")

# Use tool decorator to define the function parameters
@tool(args_schema=WeatherInput)
def get_weather(location: str, unit: str = "celsius") -> str:
    """Get the current weather for a location."""
    return f"Weather in {location}: 22°{unit}" # it would be more useful to have the temperature value as the second argument

# Usage
result = get_weather.invoke({"location": "New York", "unit": "fahrenheit"})
print(result)  # "Weather in New York: 22°fahrenheit"

Weather in New York: 22°fahrenheit


## Back to example from lesson

I used the following prompt in cursor to rewrite the code so that it is up to date with the latest version of langchain:

Please rewrite this code using JsonOutputParser and a pydantic model (copy and paste cell block here that I want to update)


In [41]:
# Define Pydantic model instead of ResponseSchema objects
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class ProductReview(BaseModel):
    gift: bool = Field(description="Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.")
    delivery_days: int = Field(description="How many days did it take for the product to arrive? If this information is not found, output -1.")
    price_value: str = Field(description="Extract any sentences about the value or price, and output them as a comma separated Python list.")

# Create JsonOutputParser with the Pydantic model
output_parser = JsonOutputParser(pydantic_object=ProductReview)

In [ ]:
# Using the JsonOutputParser defined above with the Pydantic model
# Get and print the format instructions for the prompt 
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"gift": {"description": "Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.", "title": "Gift", "type": "boolean"}, "delivery_days": {"description": "How many days did it take for the product to arrive? If this information is not found, output -1.", "title": "Delivery Days", "type": "integer"}, "price_value": {"description": "Extract any sentences about the value or price, and output them as a comma separated Python list.", "title": "Price Value", "type": "string"}}, "required": ["gif

In [ ]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)

print(messages[0].content)
response = chat(messages)
print(response.content)
output_dict = output_parser.parse(response.content)
output_dict
type(output_dict)
output_dict.get('delivery_days')

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# creating template
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

# create prompt template using template
prompt = ChatPromptTemplate.from_template(template=review_template_2)

# creating prompt
messages = prompt.format_messages(
    text=customer_review,
    format_instructions=format_instructions
)

# call chatgpt with prompt
response = chat.invoke(messages)  # prefer .invoke over __call__
print('This is the content of the response variable:')
print(response.content, '\n')

# Parse into a validated Python object (Pydantic model)
parsed = output_parser.parse(response.content) # it's good to paerse into a dict because then you can extract values in the dict

print('This is the content of the parsed output:')
print(parsed, '\n')
print(type(parsed))  

# validate into your pydantic model to then be able to use it's attributes
parsed = ProductReview.model_validate(parsed)  
print(parsed.delivery_days, '\n')         # access as attribute
print(parsed.model_dump())        # how to access all the parsed data as a dictionary

This is the content of the response variable:
{
  "gift": false,
  "delivery_days": 2,
  "price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
} 

This is the content of the parsed output:
{'gift': False, 'delivery_days': 2, 'price_value': ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]} 

<class 'dict'>
2 

{'gift': False, 'delivery_days': 2, 'price_value': ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}
